In [ ]:
import os
import pandas as pd
import cbsodata
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from dotenv import load_dotenv


In [ ]:
CATALOG_URL = 'dataderden.cbs.nl'
DATABASE = 'CBS'
SCHEMA = 'RAW'

# Alle MLZ tabellen: tabel_id -> snowflake tabel naam
TABLES = {
    '40083NED': 'WLZ_ZORG_IN_NATURA',
    '40098NED': 'WLZ_INDICATIE_STAND',
    '40103NED': 'WLZ_INDICATIE_INSTROOM',
    '40074NED': 'WLZ_EIGEN_BIJDRAGE',
    '40073NED': 'WMO_EIGEN_BIJDRAGE',
    '50118NED': 'REGIOBEELD',
}

In [ ]:
def haal_cbs_tabel(tabel_id: str) -> pd.DataFrame:
    """Haal een tabel op uit de CBS OData API en retourneer deze als een pandas DataFrame."""
    df = pd.DataFrame(cbsodata.get_data(tabel_id, catalog_url=CATALOG_URL))
    df.colums = [c.upper() for c in df.columns]
    return df

In [ ]:
def maak_snowflake_verbinding() -> snowflake.connector.SnowflakeConnection:
    """Maak een connectie met Snowflake op basis van omgevingsvariabelen."""
    load_dotenv()  # Laad omgevingsvariabelen uit .env bestand
    return snowflake.connector.connect(
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
        database=DATABASE,
        schema=SCHEMA
    )


In [ ]:
def schrijf_naar_snowflake(conn, df: pd.DataFrame, tabel_naam: str) -> None:
    """Schrijf een pandas DataFrame naar een Snowflake tabel."""
    df = df.astype({col: 'str' for col in df.select_dtypes(include='object').columns})
    write_pandas(
        conn=conn,
        df=df,
        table_name=tabel_naam,
        database=DATABASE,
        schema=SCHEMA, 
        auto_create_table=True,
        overwrite=True
    )


In [ ]:
def laad_alle_tabellen() -> None:
    """Laad alle tabellen uit de TABLES mapping in Snowflake."""
    conn = maak_snowflake_verbinding()
    try:
        for tabel_id, tabel_naam in TABLES.items():
            print(f"Haal tabel {tabel_id} op...")
            df = haal_cbs_tabel(tabel_id)
            print(f"Schrijf tabel {tabel_id} naar Snowflake als {tabel_naam}...")
            schrijf_naar_snowflake(conn, df, tabel_naam)
    finally:
        conn.close()

In [ ]:
laad_alle_tabellen()